# 06 — Formal held-out test

Run this after the final expanded probabilistic algorithm is locked using validation.
This notebook reports only the formal test split. Validation rows are included
only as support for validation-tuned selectors, and are filtered out before final summaries.

In [ ]:
from pathlib import Path
import os
import sys
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / "src"))

pd.set_option("display.max_columns", 180)
pd.set_option("display.width", 240)
print("PROJECT_ROOT:", PROJECT_ROOT, flush=True)

In [ ]:
from carnivore_reconstruction.timing import TimerLog, status
from carnivore_reconstruction.formal_large_felid import read_table_auto, summarize_with_linear, save_metric_bundle, prepare_formal_task_tables, balanced_sample_task_table
from carnivore_reconstruction.proposed import paper_metrics_for_reporting
from carnivore_reconstruction.transfer import transfer_scenario_counts

MODEL_DIR = PROJECT_ROOT / "outputs" / "pretrained_model"
MODEL_PATH = MODEL_DIR / "pretrained_model.joblib"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "formal_heldout_test"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

VALIDATION_SPLIT = "validation"
TEST_SPLIT = "test"
MAX_TEST_TASKS = 1000
MAX_VALIDATION_SUPPORT_TASKS = 800
RANDOM_SEED = 42
KEEP_SCORED = False

timer = TimerLog()

In [ ]:
from carnivore_reconstruction import MotifReconstructionModel, tasks_from_tables

status("Held-out test 1/5: loading frozen model and formal task table")
with timer.step("load_model_and_task_tables"):
    model = MotifReconstructionModel.load(MODEL_PATH)
    AUTO_N_JOBS = int(os.environ.get("CARNIVORE_N_JOBS", max(1, (os.cpu_count() or 2) - 1)))
    model.config.n_jobs = AUTO_N_JOBS
    model.config.parallel_threshold_tasks = 1
    model.config.evaluate_environmental_exposure = False
    model.config.save_candidate_paths = False
    print(f'Using n_jobs={AUTO_N_JOBS}; environmental exposure/path-row saving disabled for evaluation speed.', flush=True)
    task_table = read_table_auto(MODEL_DIR / "task_table.parquet")
    task_points = read_table_auto(MODEL_DIR / "task_points.parquet")
    split_table_path = MODEL_DIR / "formal_individual_split_table.csv"
    split_table = pd.read_csv(split_table_path) if split_table_path.exists() else None
    task_table, task_points, split_table = prepare_formal_task_tables(
        task_table,
        task_points,
        split_table,
        seed=20260625,
        label="formal_task_table",
    )

print("task_table:", task_table.shape)
display(task_table.groupby(["dataset", "taxon", "setting_name", "split"]).size().reset_index(name="n_tasks"))

print("Train to test transfer scenario counts:")
display(transfer_scenario_counts(task_table, source_split="train", target_split=TEST_SPLIT, exclude_same_animal=True).head(30))

In [ ]:
status("Held-out test 2/5: selecting test tasks with validation support")
test_table = task_table[task_table["split"].eq(TEST_SPLIT)].copy().reset_index(drop=True)
if test_table.empty:
    raise ValueError("No formal test tasks found. Check the individual-disjoint split.")
test_table = balanced_sample_task_table(test_table, MAX_TEST_TASKS, seed=RANDOM_SEED, label="formal_heldout_test")

validation_table = task_table[task_table["split"].eq(VALIDATION_SPLIT)].copy().reset_index(drop=True)
validation_table = balanced_sample_task_table(validation_table, MAX_VALIDATION_SUPPORT_TASKS, seed=RANDOM_SEED, label="validation_support_for_test")

# Existing overnight pretrained models may not yet contain ecological candidate artifacts.
from carnivore_reconstruction.ecological_candidates import ensure_ecological_artifacts
train_table_for_artifacts = balanced_sample_task_table(task_table[task_table["split"].eq("train")].copy(), 3000, seed=RANDOM_SEED, label="artifact_training_support")
train_tasks_for_artifacts = tasks_from_tables(train_table_for_artifacts, task_points)
model = ensure_ecological_artifacts(model, train_tasks_for_artifacts, timer=timer)
evaluation_table = pd.concat([validation_table, test_table], ignore_index=True, sort=False).drop_duplicates("task_uid")
evaluation_table = evaluation_table.sort_values(["split", "dataset", "taxon", "setting_name", "animal_id", "task_uid"]).reset_index(drop=True)

evaluation_tasks = tasks_from_tables(evaluation_table, task_points)
test_tasks = tasks_from_tables(test_table, task_points)
test_uids = set(test_table["task_uid"].astype(str))

print(f"Selected {len(test_tasks):,} formal held-out test tasks")
print("Evaluation rows by split:")
print(evaluation_table["split"].value_counts(dropna=False).to_string())

In [ ]:
from carnivore_reconstruction.benchmark import run_accuracy_benchmark

status("Held-out test 3/5: running validation-supported publication test")
with timer.step("run_formal_heldout_test", n_tasks=len(evaluation_tasks), split=f"{VALIDATION_SPLIT}+{TEST_SPLIT}"):
    result = run_accuracy_benchmark(model, evaluation_tasks, keep_scored=KEEP_SCORED, task_table=task_table)

all_task_metrics = result["task_metrics"]
all_selected_candidates = result.get("selected_candidates", pd.DataFrame())
all_runtime = result.get("runtime", pd.DataFrame())
setting_choices = result.get("setting_choices", pd.DataFrame())

def _test_only(df):
    if df is None or df.empty or "task_uid" not in df.columns:
        return pd.DataFrame() if df is None else df
    return df[df["task_uid"].astype(str).isin(test_uids)].copy().reset_index(drop=True)

task_metrics = _test_only(all_task_metrics)
selected_candidates = _test_only(all_selected_candidates)
runtime = _test_only(all_runtime)

print("all metrics:", all_task_metrics.shape)
print("test metrics:", task_metrics.shape)
print("test runtime:", runtime.shape)

In [ ]:
status("Held-out test 4/5: saving formal held-out test outputs")
with timer.step("save_formal_heldout_test_outputs", n_tasks=len(test_tasks)):
    all_task_metrics.to_csv(OUTPUT_DIR / "formal_heldout_test_all_eval_task_metrics_validation_plus_test.csv", index=False)
    all_runtime.to_csv(OUTPUT_DIR / "formal_heldout_test_all_eval_runtime_validation_plus_test.csv", index=False)
    save_metric_bundle(task_metrics, OUTPUT_DIR, "formal_heldout_test", paper_filter_func=paper_metrics_for_reporting)
    selected_candidates.to_csv(OUTPUT_DIR / "formal_heldout_test_selected_candidates.csv", index=False)
    runtime.to_csv(OUTPUT_DIR / "formal_heldout_test_runtime.csv", index=False)
    setting_choices.to_csv(OUTPUT_DIR / "formal_heldout_test_setting_choices.csv", index=False)

    report = [
        "Formal large-felid held-out test report",
        "======================================",
        f"n_test_tasks: {len(test_tasks)}",
        f"n_validation_support_tasks: {len(validation_table)}",
        "",
        "Paper method summary:",
        pd.read_csv(OUTPUT_DIR / "paper_formal_heldout_test_method_summary.csv").to_string(index=False),
    ]
    (OUTPUT_DIR / "formal_heldout_test_report.txt").write_text("\n".join(report), encoding="utf-8")
    timer.save(OUTPUT_DIR / "runtime_formal_heldout_test.csv")

print("Saved outputs to:", OUTPUT_DIR)
display(pd.read_csv(OUTPUT_DIR / "paper_formal_heldout_test_method_summary.csv"))

In [ ]:
status("Held-out test 5/5: quick runtime summary")
if not runtime.empty:
    time_cols = [c for c in runtime.columns if c.endswith("seconds") or c.endswith("_seconds")]
    if time_cols:
        display(runtime[time_cols].describe(percentiles=[0.5, 0.75, 0.90, 0.95]).T)